In [1]:
%pip install nilearn nibabel packaging --upgrade


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
"""
Figure 4 (panel d): affordance landscape modes visualized on the cortical surface.

"""

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from scipy.linalg import eig
from IPython.display import display

!pip install nilearn nibabel
import nibabel as nib
from nilearn import plotting, datasets, image
from nilearn.surface import vol_to_surf

# --- paths ---------------------------------------------------------------
DATA_DIR = "HCP_Schaefer400_structural_conn_matrices"
ATLAS_TSV = "atlas-Schaefer2018v0143_desc-400ParcelsAllNetworks_dseg.tsv"
OUT_DIR = "results/fig4"
os.makedirs(OUT_DIR, exist_ok=True)

mpl.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}",
})

T = 1.0  # time horizon
N_MODES_PER_SIDE = 4  # top-4 easy, top-4 hard

matrices = {}
for file in sorted(os.listdir(DATA_DIR)):
    if file.endswith(".csv"):
        subject_id = file.replace(".csv", "")
        matrices[subject_id] = pd.read_csv(os.path.join(DATA_DIR, file), header=None).values

print(f"Loaded {len(matrices)} subjects")

atlas = pd.read_csv(ATLAS_TSV, sep="\t")
atlas["network"] = atlas["network_label"]  # Yeo-7 label
networks_7 = atlas["network"].unique()
network_indices_7 = {net: atlas.index[atlas["network"] == net].tolist() for net in networks_7}

all_subnetwork_matrices = {
    subj: {net: mat[np.ix_(idx, idx)] for net, idx in network_indices_7.items()}
    for subj, mat in matrices.items()
}

print(f"Available Yeo-7 network keys: {list(networks_7)}")

# Load fsaverage surface + Schaefer atlas volume once, shared across networks
print("Fetching fsaverage + Schaefer atlas...")
fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")
schaefer = datasets.fetch_atlas_schaefer_2018(n_rois=400, resolution_mm=1)
atlas_img = nib.load(schaefer["maps"])
atlas_data = atlas_img.get_fdata()


def normalize(mode):
    return mode / np.max(np.abs(mode))


def make_mode_img(mode_weights, region_indices):
    mode_full = np.zeros(400)
    mode_full[region_indices] = mode_weights
    mode_img_data = np.zeros(atlas_data.shape)
    for i, val in enumerate(mode_full):
        mode_img_data[atlas_data == (i + 1)] = val
    mode_img = image.new_img_like(atlas_img, mode_img_data)
    lh = vol_to_surf(mode_img, fsaverage["pial_left"])
    rh = vol_to_surf(mode_img, fsaverage["pial_right"])
    return mode_img, lh, rh


def plot_surface_view(hemi, view, surf_mesh, surf_data_list, mode_titles, all_energies,
                       view_label, hemi_label, net_name, net_key, subject):
    """2x4 grid of easy/hard modes for a given hemisphere/view."""
    fig, axes = plt.subplots(2, 4, figsize=(22, 10), subplot_kw={"projection": "3d"})

    for i, (surf_data, title, energy) in enumerate(zip(surf_data_list, mode_titles, all_energies)):
        ax = axes[i // 4, i % 4]
        plotting.plot_surf_stat_map(
            surf_mesh=surf_mesh, stat_map=surf_data, hemi=hemi, view=view,
            bg_map=fsaverage[f"sulc_{hemi}"], bg_on_data=True, colorbar=False,
            cmap="RdBu", vmax=1.0, vmin=-1.0, axes=ax,
            title=rf"{title} --- $W^{{-1}}={energy:.1e}$",
        )

    sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=-1, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes.ravel().tolist(), orientation="vertical",
                         fraction=0.015, pad=0.01, shrink=0.5)
    cbar.set_label(r"Normalised mode weight", fontsize=12)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    cbar.ax.tick_params(labelsize=10)

    view_tag = f"{hemi_label}{view_label}"
    fig.suptitle(rf"{net_key} Modes --- {hemi_label.capitalize()} {view_label.capitalize()} "
                 rf"(Subject: {subject})", fontsize=13, y=1.01)
    out_path = f"{OUT_DIR}/{net_name}_{view_tag}.pdf"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    fig.canvas.draw()
    display(fig)
    plt.close()
    print(f"Saved: {out_path}")


def plot_network_modes(net_name, subject=None, n_modes_per_side=N_MODES_PER_SIDE):
    """
    Plot the n_modes_per_side lowest-cost ('easy') and highest-cost ('hard')
    affordance-landscape modes for a Yeo-7 network, on the cortical surface

    net_name : substring matching a Yeo-7 network key, e.g. 'Vis', 'Default'.
    subject  : subject ID to use; defaults to the first subject loaded.
    """
    if subject is None:
        subject = list(all_subnetwork_matrices.keys())[0]

    net_keys = list(all_subnetwork_matrices[subject].keys())
    net_key = [k for k in net_keys if net_name in k][0]
    print(f"Using network key: '{net_key}'")

    A = all_subnetwork_matrices[subject][net_key].astype(float)
    net_indices = atlas.index[atlas["network"] == net_name].tolist()

    lam, V = eig(A)
    with np.errstate(divide="ignore", invalid="ignore"):
        W_lam = np.where(np.abs(lam) > 1e-12, (np.exp(2 * lam * T) - 1) / (2 * lam), T)
    W_lam_inv = 1.0 / np.abs(W_lam.real)

    sorted_idx = np.argsort(W_lam_inv)
    easy_indices = sorted_idx[:n_modes_per_side]
    hard_indices = sorted_idx[-n_modes_per_side:]
    mode_indices = list(easy_indices) + list(hard_indices)
    mode_titles = ([rf"Easy {i+1}" for i in range(n_modes_per_side)] +
                   [rf"Hard {i+1}" for i in range(n_modes_per_side)])

    # --- project each mode onto the cortical surface ---
    print("Computing surface projections...")
    all_lh, all_rh, all_energies = [], [], []
    for mode_idx in mode_indices:
        mode_raw = normalize(V[:, mode_idx].real)
        _, lh_data, rh_data = make_mode_img(mode_raw, net_indices)
        all_lh.append(lh_data)
        all_rh.append(rh_data)
        all_energies.append(W_lam_inv[mode_idx])
    print("Done.")

    views = [
        ("left", "medial", fsaverage["infl_left"], all_lh, "left", "medial"),
        ("left", "lateral", fsaverage["infl_left"], all_lh, "left", "lateral"),
        ("right", "medial", fsaverage["infl_right"], all_rh, "right", "medial"),
        ("right", "lateral", fsaverage["infl_right"], all_rh, "right", "lateral"),
    ]
    for hemi, view, surf_mesh, surf_data_list, hemi_label, view_label in views:
        print(f"\nPlotting {hemi_label} {view_label}...")
        plot_surface_view(hemi, view, surf_mesh, surf_data_list, mode_titles, all_energies,
                           view_label, hemi_label, net_name, net_key, subject)

    # --- ortho view summary ---
    fig = plt.figure(figsize=(24, 10))
    gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 1, 0.05], hspace=0.35, wspace=0.15)

    for i, (mode_idx, title) in enumerate(zip(mode_indices, mode_titles)):
        ax = fig.add_subplot(gs[i // 4, i % 4])
        mode_raw = normalize(V[:, mode_idx].real)
        energy = W_lam_inv[mode_idx]
        mode_img, _, _ = make_mode_img(mode_raw, net_indices)

        plotting.plot_stat_map(
            mode_img, colorbar=False, cmap="RdBu", vmax=1.0,
            display_mode="ortho", annotate=False, axes=ax,
            title=rf"{title} --- $W^{{-1}}={energy:.1e}$",
        )
        for child_ax in fig.axes:
            for line in child_ax.lines:
                line.set_alpha(0.15)
                line.set_linewidth(0.3)

    cbar_ax = fig.add_subplot(gs[:, 4])
    sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=-1, vmax=1))
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label(r"Normalised mode weight", fontsize=12)
    cbar.set_ticks([-1, -0.5, 0, 0.5, 1])
    cbar.ax.tick_params(labelsize=10)

    fig.suptitle(rf"{net_key} Modes --- Ortho View (Subject: {subject})", fontsize=13, y=1.01)
    out_path = f"{OUT_DIR}/{net_name}_modes_ortho.pdf"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    fig.canvas.draw()
    display(fig)
    plt.close()
    print(f"Saved: {out_path}")

    return {"net_key": net_key, "mode_indices": mode_indices, "energies": all_energies}


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Loaded 100 subjects
Available Yeo-7 network keys: ['Vis', 'SomMot', 'DorsAttn', 'SalVentAttn', 'Limbic', 'Cont', 'Default']
Fetching fsaverage + Schaefer atlas...
[fetch_atlas_schaefer_2018] Dataset directory found: /Users/suman/nilearn_data/schaefer_2018


In [ ]:
plot_network_modes("Default")

In [ ]:
plot_network_modes("Vis")